#### Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#### Define functions

In [ ]:

def noise_speeds(detector_df, speed_error_df, rand_seed=0):
    """ 
    Function to apply noise to detector speeds by randomly sampling percentage errors from the speed_error_df.
    Returns the updated detector_df with a new column 'vPKW_noised' containing the noised speeds.
    """

    # Initialize the seed for reproducibility
    np.random.seed(rand_seed)

    # Get 'pct_error' values from count_df and convert to array for sampling
    speed_pct_error_pool = speed_error_df['pct_error'].values

    # Determine how many noise values we need to sample.
    num_samples = len(detector_df)

    # Randomly sample noise values w/ replacement
    sampled_noise = np.random.choice(speed_pct_error_pool, size=num_samples, replace=True)
    detector_df['pct_error'] = sampled_noise

    # Calculate noise
    detector_df['vPKW_noised'] = (detector_df['vPKW'] * (1 + detector_df['pct_error'] / 100)).round(2)

    return detector_df

def sample_and_apply_error_to_15min_interval_groupings(detector_df, count_error_df, rand_seed=0):
    """
    Function to sample noise from the count_error_df and apply it to the detector_df.
    Noise in the count_error_df is for 15-minute, per-detector intervals, so we sample and apply
    the percentage errors to the detector_df at this same granularity. Later, we'll distribute the noised values 
    to the 30-second intervals that the original detector_df contains.

    Args:
        detector_df (pd.DataFrame): DataFrame containing detector data with 'Detector', 'Time', and 'qPKW' columns.
        count_error_df (pd.DataFrame): DataFrame containing count noise data with 'pct_error' column.
        rand_seed (int): Random seed for reproducibility.
    Returns:
        pd.DataFrame: Updated detector_df with noisy 15-minute sums and proportions.
    """
    # Initialize the seed for reproducibility
    np.random.seed(rand_seed)

    # Get 'pct_error' values from count_df and convert to array for sampling
    count_pct_error_pool = count_error_df['pct_error'].values

    # Create a 15-minute interval start column for grouping in detector_df
    detector_df['15_min_interval_start'] = (detector_df['Time'] // 15) * 15

    # Create a DataFrame for sampling, ensuring we have unique combinations of Detector and 15_min_interval_start
    sampling_df = detector_df.drop_duplicates(subset=['Detector', '15_min_interval_start'])[['Detector', '15_min_interval_start']]

    # Determine how many noise values we need to sample.
    num_samples = len(sampling_df)

    # Randomly sample noise values w/ replacement
    sampled_noise = np.random.choice(count_pct_error_pool, size=num_samples, replace=True)

    sampling_df['pct_error'] = sampled_noise

    # Merge pct_error into detector_df based on both time interval and detector_id
    detector_df = pd.merge(detector_df, sampling_df, on=['Detector', '15_min_interval_start'], how='left')

    # Calculate original 15-minute sums and proportions
    detector_df['original_15min_qPKW_sum'] = detector_df.groupby(['Detector', '15_min_interval_start'])['qPKW'].transform('sum')

    # Calculate original proportion of each 30-sec count within its 15-min sum
    # Handle division by zero if original_15min_qPKW_sum is 0
    detector_df['original_30sec_proportion'] = detector_df.apply(
        lambda row: row['qPKW'] / row['original_15min_qPKW_sum'] if row['original_15min_qPKW_sum'] > 0 else 0,
        axis=1
    )

    # Calculate floating-point noisy 15-minute sums
    detector_df['qPKW_15min_float_noisy'] = detector_df['original_15min_qPKW_sum'] * (1 + detector_df['pct_error'] / 100)

    # Apply standard rounding to get integer noisy 15-minute sums
    # Group by both interval and detector_id before transforming
    detector_df['qPKW_15min_noisy_int'] = detector_df.groupby(['Detector', '15_min_interval_start'])['qPKW_15min_float_noisy'].transform(
        lambda x: int(np.round(x.iloc[0])) # Apply standard rounding once per group
    )

    # Ensure the total is non-negative
    detector_df['qPKW_15min_noisy_int'] = detector_df['qPKW_15min_noisy_int'].apply(lambda x: max(0, x))

    return detector_df

def noise_counts_by_distributing_difference(detector_df, rand_seed=0):
    """
    Function to apply noise to 30-second counts by distributing the difference, where the difference is the
    difference between the noisy 15-minute total and the original 15-minute sum. 
    Distribution occurs based on the original proportions of each 30-second count within its 15-minute sum.
    
    E.g., if the original 15-minute sum is 100, error is 10%, and the noisy 15-minute total is 110, then we randomly (but proportionally) distribute 10 additional vehicles among the 
    thirty total 30-second counts that comprise the 15-minute interval. If the original sum is 100, error is -10%, and the noisy total is 90,
    then we randomly (but proportionally) remove 10 vehicles from the thirty total 30-second counts that comprise the 15-minute interval.

    This function modifies the input DataFrame and returns it, adding a new column 'qPKW_noised' with the noised 30-second counts.
    """
    # Initialize the seed for reproducibility
    rng = np.random.RandomState(rand_seed)

    detector_df['qPKW_noised'] = 0 # Initialize new noised column

    # Iterate over each unique (15-minute interval, detector_id) combination
    for (interval_start, detector_id), group in detector_df.groupby(['Detector', '15_min_interval_start']):
        noisy_15min_total = group['qPKW_15min_noisy_int'].iloc[0] # The target sum
        original_15min_sum = group['original_15min_qPKW_sum'].iloc[0] # The original sum for this group
        original_qPKW_30sec_in_interval = group['qPKW'].values # Original 30-sec counts for this group

        net_change = noisy_15min_total - original_15min_sum
        
        num_slots = len(original_qPKW_30sec_in_interval)
        # Start with original values and apply changes
        new_30sec_counts = np.copy(original_qPKW_30sec_in_interval) 
        
        if net_change > 0:
            # Add vehicles: distribute positive difference using multinomial
            if original_15min_sum > 0:
                # Use original proportions for distribution
                normalized_proportions = original_qPKW_30sec_in_interval / original_15min_sum
                additions = rng.multinomial(n=net_change, pvals=normalized_proportions)
            else:
                # If original sum was 0, but we need to add, distribute randomly among slots
                additions = np.zeros(num_slots, dtype=int)
                if num_slots > 0: # Only if there are slots to add to
                    random_indices = rng.choice(num_slots, size=net_change, replace=True)
                    for idx in random_indices:
                        additions[idx] += 1
            
            new_30sec_counts += additions

        elif net_change < 0:
            # Remove vehicles: distribute negative difference
            num_to_remove = abs(net_change)

            # Handle cases where original sum is 0 or we try to remove more than exist
            if original_15min_sum <= 0 or original_15min_sum < num_to_remove:
                new_30sec_counts = np.zeros(num_slots, dtype=int)
            else:
                # Create a pool of indices weighted by original counts
                index_pool = []
                for idx, count in enumerate(original_qPKW_30sec_in_interval):
                    index_pool.extend([idx] * count)
                
                # Randomly select indices to remove (without replacement)
                # These are the "vehicles" that will be removed from their respective slots
                removed_indices = rng.choice(index_pool, size=num_to_remove, replace=False)
                
                # Count how many times each original 30-second slot's index was chosen for removal
                # This directly tells us how many vehicles to subtract from each slot
                removals_count_per_index = np.bincount(removed_indices, minlength=num_slots)
                
                # Subtract removals from original counts
                new_30sec_counts = original_qPKW_30sec_in_interval - removals_count_per_index
                new_30sec_counts[new_30sec_counts < 0] = 0 # Ensure no negative counts if by chance some bins went below zero
                                                        # (should be rare with `replace=False` unless error in logic)

        # Ensure final counts are non-negative (redundant but safe)
        new_30sec_counts[new_30sec_counts < 0] = 0 
        
        # Assign the new noised 30-second counts back to the DataFrame for this specific group
        detector_df.loc[group.index, 'qPKW_noised'] = new_30sec_counts

    return detector_df

def noise_counts_by_total_redistribution(detector_df, rand_seed=0):
    """
    Function to apply noise to 30-second counts by redistributing the entire noised 15-minute total across the 30-second intervals.
    Distribution occurs based on the original proportions of each 30-second count within its 15-minute sum.
    
    E.g., if the original 15-minute sum is 100, error is 10%, and the noisy 15-minute total is 110, then we randomly (but proportionally) redistribute all 110 vehicles among the
    thirty total 30-second counts that comprise the 15-minute interval. If the original sum is 100, error is -10%, and the noisy total is 90,
    then we randomly (but proportionally) redistribute all 90 vehicles among the thirty total 30-second counts that comprise the 15-minute interval.
    
    This function modifies the input DataFrame and returns it, adding a new column 'qPKW_noised' with the noised 30-second counts.
    """
    
    # Initialize the seed for reproducibility
    rng = np.random.RandomState(rand_seed)

    detector_df['qPKW_noised'] = 0 # Initialize new noised column

    # Iterate over each unique (15-minute interval, detector_id) combination
    for (interval_start, detector_id), group in detector_df.groupby(['Detector', '15_min_interval_start']):
        noisy_15min_total = group['qPKW_15min_noisy_int'].iloc[0] # Get the single noisy total for this group
        original_proportions = group['original_30sec_proportion'].values
        
        # Handle cases where original sum was zero but noisy sum is non-zero
        # Or where original sum was non-zero but noisy sum became zero (e.g., due to negative error)
        if noisy_15min_total > 0 and np.sum(original_proportions) > 0:
            # Normalize proportions just in case (should sum to 1 if original_15min_sum > 0)
            normalized_proportions = original_proportions / np.sum(original_proportions)
            
            # Distribute the noisy total among the 30-sec intervals using multinomial distribution
            new_30sec_counts = rng.multinomial(n=noisy_15min_total, pvals=normalized_proportions)
        
        elif noisy_15min_total > 0 and np.sum(original_proportions) == 0:
            # Handle case where original 15-min sum was 0, but noise made it > 0.
            # Distribute value randomly among the available slots
            num_slots = len(original_proportions)
            new_30sec_counts = np.zeros(num_slots, dtype=int)
            if num_slots > 0: # Ensure there are slots to put values into
                random_indices = rng.choice(num_slots, size=noisy_15min_total, replace=True)
                for idx in random_indices:
                    new_30sec_counts[idx] += 1
            
        else:
            # Handle scenario where noisy_15min_total is 0, so all 30-sec counts are 0
            new_30sec_counts = np.zeros_like(original_proportions, dtype=int)
        
        # Assign the new noised 30-second counts back to the DataFrame
        detector_df.loc[group.index, 'qPKW_noised'] = new_30sec_counts

    return detector_df

def verify_df(detector_df):
    """
    Function to verify that the disaggregated noisy 30-second counts sum to the noised 15-minute totals.
    The 'noised_15min_int_total' and 'disaggregated_noised_sum' columns should match,
    """
    verification_df = detector_df.groupby(['Detector', '15_min_interval_start']).agg(
        original_sum=('qPKW', 'sum'),
        pct_error=('pct_error', 'first'), # Display error for context
        noised_15min_int_total=('qPKW_15min_noisy_int', 'first'), # Should be same for all rows in group
        disaggregated_noised_sum=('qPKW_noised', 'sum')
    )
    
    print("Verification DatFrame -- the 'noised_15min_int_total' and 'disaggregated_noised_sum' columns should match:")
    
    print(verification_df.head())

    return verification_df

def plot_pct_errors(detector_df, filepath, figsize=(8,3), fontsize=8):
    """
    Function to plot errors for comparison. 
    """
    # Calculate pct errors between 30-sec noised counts and original counts
    detector_df['30sec_pct_error'] = (detector_df['qPKW_noised'] / detector_df['qPKW'] - 1) * 100
    # Handle NaNs
    detector_df['30sec_pct_error'] = detector_df['30sec_pct_error'].fillna(0)

    # Calculate empirical mean of each detector/15min-interval grouping's 30-second-errors (thus each mean is computed across 30 vals)
    # This can be compared to the original pct_error values, since they are sampled per etector/15min-interval grouping
    empirical_grouping_error_df = detector_df.groupby(['Detector', '15_min_interval_start'])['30sec_pct_error'].mean().reset_index()

    # Calculate mean of the original pct error values per detector-interval grouping -- these should be same as the sampled values initially, since pct_error values 
    # are applied to each detector/15min-interval grouping
    orig_grouping_error_df = detector_df.groupby(['Detector', '15_min_interval_start'])['pct_error'].mean().reset_index()

    # Create the figure w/ 2 subplots
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=figsize) 

    # Left Subplot: Error comparison at detector/15-min interval level
    axes[0].hist(empirical_grouping_error_df['30sec_pct_error'], bins=50, alpha=0.7, label='Empirical per-grouping means of 30-sec pct errors')
    axes[0].hist(orig_grouping_error_df['pct_error'], bins=50, alpha=0.7, label='Sampled applied pct errors')
    axes[0].set_title("Error comparison at detector/15-min interval level", fontsize=fontsize)
    axes[0].set_xlabel("15-min mean pct errors (%)", fontsize=fontsize)
    axes[0].set_ylabel("Frequency", fontsize=fontsize)
    axes[0].legend(fontsize=fontsize)
    axes[0].grid(axis='y', alpha=0.75)

    # Right Subplot: Individual 30-sec pct errors across all detectors and time
    axes[1].hist(detector_df['30sec_pct_error'], bins=50, alpha=0.7, color='blue')
    axes[1].set_title("Individual 30-sec pct errors (across all detectors and time)", fontsize=fontsize)
    axes[1].set_xlabel("30-sec pct errors (%)", fontsize=fontsize)
    # axes[1].set_ylabel("Frequency", fontsize=fontsize)
    axes[1].grid(axis='y', alpha=0.75)

    # Display plot
    plt.tight_layout()
    plt.suptitle(f'Percentage Error Comparisons:\n{filepath}', y=1.02, fontsize=fontsize) # Overall title for the figure
    plt.show()
    


#### Load in error data

In [ ]:
## Load in all count noise data into a single DF
count_dir_path = "noise_data/count_data"
count_col_names = ['timestamp', 'pct_error']

# Get a list of all CSV file paths
csv_files = [os.path.join(count_dir_path, file) for file in os.listdir(count_dir_path) if file.endswith(".csv")]

# Load all CSV files into a single DataFrame
count_error_df = pd.concat([pd.read_csv(file_path, header=None, names=count_col_names) for file_path in csv_files], ignore_index=True)


In [ ]:
## Load in speed noise data into a DF
speed_dir_path = "noise_data/speed_data"
speed_col_names = ['rtms_speed_mph', 'microloop_speed_mph']

# Get a list of all CSV file paths
csv_files = [os.path.join(speed_dir_path, file) for file in os.listdir(speed_dir_path) if file.endswith(".csv")]

# Load all CSV files into a single DataFrame
speed_error_df = pd.concat([pd.read_csv(file_path, header=None, names=speed_col_names) for file_path in csv_files], ignore_index=True)

# Calculate % error of microloop speed relative to RTMS baseline
speed_error_df['pct_error'] = (speed_error_df['microloop_speed_mph'] / speed_error_df['rtms_speed_mph'] - 1) * 100

#### Define variables

(distribute_difference, rather than distribute_total, was selected for the original benchmarking work)

In [ ]:
compare_count_distribution_methods = False    # Used for comparison. Saving is disabled when this is True.
verify_dfs = True   # Prints verification DFs to console so user can check that noise is correctly applied.
gen_plots = True    # Generates comparison plots
save_outputs = True    # Saves the noised detector data to disk

assert not (compare_count_distribution_methods and save_outputs), "Cannot save outputs when comparing distribution methods!"

if not compare_count_distribution_methods:
    distribute_difference = True        # Distributes only the difference between noisy 15-min total and original 15-min sum among 30-sec counts
    distribute_total = False            # Redistributes the entire noisy 15-min total across 30-sec counts
    assert distribute_difference != distribute_total, "One -- and only one -- distribution method can be selected at a time!"


detections_dir_path = "detector_measurements"
filepaths_to_noise = [
    "1121/detections_0420-0540.csv",    # Scenario 1: Train
    "1122/detections_0420-0540.csv",
    "1123/detections_0420-0540.csv",
    "1124/detections_0420-0540.csv",    # Scenario 1: Test
    "1128/detections_0420-0540.csv",
    "1129/detections_0420-0540.csv",
    "1121/detections_0360-0600.csv",    # Scenario 2: Train
    "1122/detections_0360-0600.csv",
    "1123/detections_0360-0600.csv",
    "1129/detections_0360-0600.csv",    # Scenario 2: Test
    "1201/detections_0360-0600.csv",
    "1202/detections_0360-0600.csv",
    "1121/detections_0360-0600.csv",    # Scenario 3: Train
    "1122/detections_0360-0600.csv",
    "1123/detections_0360-0600.csv",
    "1128/detections_0360-0600.csv",    # Scenario 3: Test
    "1129/detections_0360-0600.csv",
    "1130/detections_0360-0600.csv",
    "1201/detections_0360-0600.csv",    
    "1202/detections_0360-0600.csv",
    ]


#### Load in detection data and apply noise

In [ ]:
# Iterate through each file and apply noise
for i, filepath in enumerate(filepaths_to_noise):
    
    # Load in detector data
    full_path = os.path.join(detections_dir_path, filepath)
    detector_df = pd.read_csv(full_path, sep=';')

    # Apply noise to the detector speeds
    detector_df = noise_speeds(detector_df, speed_error_df, rand_seed=i)
    # Since we want the noise-applied column, replace accordingly
    detector_df['vPKW'] = detector_df['vPKW_noised']
    # Keep only the relevant columns
    detector_df = detector_df[['Detector', 'Time', 'qPKW', 'vPKW']]

    # Apply noise to the detector data at 15-minute intervals (since that's count noise data we have)
    detector_df = sample_and_apply_error_to_15min_interval_groupings(detector_df, count_error_df, rand_seed=i)

    ## Distribute noise to 30-second intervals based on the 15-minute sums
    # If comparing distribution methods, apply both methods to compare
    if compare_count_distribution_methods:
        print("Distributing noise by difference...")
        detector_df = noise_counts_by_distributing_difference(detector_df, rand_seed=i)
        if verify_dfs:
            verify_df(detector_df)
        if gen_plots:
            plot_pct_errors(detector_df, filepath) 
        print("Distributing noise by total redistribution...")
        detector_df = noise_counts_by_total_redistribution(detector_df, rand_seed=i)
        if verify_dfs:
            verify_df(detector_df)
        if gen_plots:
            plot_pct_errors(detector_df, filepath) 
    
    # If not comparing methods, apply only one of the distribution methods and save the results
    else:
        if distribute_difference:
            print("Distributing noise by difference...")
            detector_df = noise_counts_by_distributing_difference(detector_df, rand_seed=i)
        elif distribute_total:
            print("Distributing noise by total redistribution...")
            detector_df = noise_counts_by_total_redistribution(detector_df, rand_seed=i)

        if verify_dfs:
            verify_df(detector_df)
        
        if gen_plots:
            plot_pct_errors(detector_df, filepath)

        if save_outputs:
            # Save the noised detector data
            noised_filepath = os.path.join(os.path.dirname(filepath), "noised_" + os.path.basename(filepath))
            save_path = os.path.join(detections_dir_path, noised_filepath)
            detector_df['qPKW'] = detector_df['qPKW_noised']
            detector_df = detector_df[['Detector', 'Time', 'qPKW', 'vPKW']]
            detector_df.to_csv(save_path, sep=';', index=False)